# Try the personal-card-agent agent

Two ways to try this repo's agent from a notebook:

- **Option A - quick local invoke.** Builds the agent in-process with the
  OSS `deepagents` library, reusing this repo's `instructions.md` and
  `middleware/` (customer-context seeding + tool-call logging). No `mda`
  CLI or server needed - fastest way to iterate.
- **Option B - talk to the real managed agent.** Connects to the agent
  exactly as `agent.py` defines it, served locally by `mda dev .`.

**Prerequisites**

```bash
uv sync --extra notebook
# fill OPENAI_API_KEY (and LANGSMITH_API_KEY for option B) in ../.env
jupyter notebook notebooks/try_agent.ipynb
```

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

# Notebook runs from notebooks/, but the agent's code (middleware/, data/,
# instructions.md, .env) lives one level up at the repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

import os

assert os.environ.get("OPENAI_API_KEY"), (
    f"Set OPENAI_API_KEY in {REPO_ROOT / '.env'} first"
)

from text_utils import extract_final_text


## Option A - quick local invoke (no `mda dev` needed)

In [6]:
from deepagents import create_deep_agent

from middleware.audit import log_tool_calls
from middleware.seed_customer_context import SeedCustomerContext

instructions = (REPO_ROOT / "instructions.md").read_text()

local_agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[],
    middleware=[log_tool_calls, SeedCustomerContext()],
    system_prompt=instructions,
)

result = local_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Thống kê số tiền tôi đã chi tiêu vào ăn uống"}
        ]
    }
)
print(extract_final_text(result["messages"][-1].content))

Calling tool: read_file
Finished tool: read_file
Dựa trên các giao dịch hiện có trong dữ liệu, giai đoạn **06/2026–08/2026** *(tháng 08 tính đến 29/08/2026)*, bạn đã chi tiêu cho **Ăn uống** như sau:

| Tháng | Các khoản ăn uống | Tổng |
|---|---:|---:|
| 06/2026 | 85.000 + 65.000 + 650.000 + 45.000 + 1.200.000 + 180.000 + 220.000 + 655.000 | **3.100.000 VNĐ** |
| 07/2026 | 90.000 + 210.000 + 480.000 + 1.350.000 + 70.000 + 250.000 + 50.000 + 950.000 | **3.450.000 VNĐ** |
| 08/2026 | 85.000 + 195.000 + 60.000 + 620.000 + 890.000 + 230.000 + 40.000 + 780.000 | **2.900.000 VNĐ** |

**Tổng chi tiêu ăn uống:**  
**3.100.000 + 3.450.000 + 2.900.000 = 9.450.000 VNĐ**

Trong toàn bộ chi tiêu thẻ giai đoạn này là **36.002.000 VNĐ**, nhóm **Ăn uống chiếm khoảng 26,2%** tổng chi tiêu. Tháng chi nhiều nhất cho ăn uống là **07/2026: 3.450.000 VNĐ**.


## Option B - talk to the managed agent via `mda dev`

In a separate terminal, from the repo root, run:

```bash
mda dev .
```

It prints the local Agent Server URL (default `http://127.0.0.1:2024`) and
the graph/assistant name to use below - it's usually the agent `name` from
`agent.py` (`"personal-card-agent"`), but check the printed Studio URL /
server logs and adjust `ASSISTANT_ID` if it differs.

In [ ]:
from langgraph_sdk import get_sync_client

SERVER_URL = "http://127.0.0.1:2024"
ASSISTANT_ID = "personal-card-agent"

client = get_sync_client(url=SERVER_URL)

thread = client.threads.create()
for chunk in client.runs.stream(
    thread["thread_id"],
    ASSISTANT_ID,
    input={
        "messages": [
            {"role": "user", "content": "Tháng này tôi đã chi tiêu bao nhiêu trên thẻ?"}
        ]
    },
    stream_mode="values",
):
    if chunk.event == "values":
        print(extract_final_text(chunk.data["messages"][-1]["content"]))

## Verification checklist (local, rows 1-9 and 11 only — row 10 needs Studio, see docs/superpowers/plans/2026-08-30-personal-card-agent-verification.md)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

verify_agent = create_deep_agent(
    model="openai:gpt-5.5",
    tools=[],
    middleware=[log_tool_calls, SeedCustomerContext()],
    system_prompt=instructions,
    checkpointer=MemorySaver(),
)

CHECKLIST_QUESTIONS = [
    "Tháng này tôi đã chi tiêu bao nhiêu trên thẻ?",
    "Giao dịch lớn nhất tuần vừa rồi là gì?",
    "Liệt kê các giao dịch trên 2 triệu trong tháng 7",
    'Danh mục nào đang "ngốn" tiền của tôi nhất?',
    "Tôi có đang trả lãi thẻ tín dụng không, và vì sao?",
    "Tôi nên dùng thẻ tín dụng hay ghi nợ cho khoản chi 5 triệu này?",
    "Ngày sao kê và ngày đến hạn thanh toán của thẻ tôi là khi nào?",
    "Tôi có đang chi tiêu vượt ngân sách tháng này không?",
    "Ngân sách của tôi là 15 triệu/tháng.",
    "Vậy tôi có đang chi tiêu vượt ngân sách tháng này không?",
    "Khoá thẻ của tôi giúp tôi.",
]
THREAD_CONFIG = {"configurable": {"thread_id": "notebook-verify-1"}}

for question in CHECKLIST_QUESTIONS:
    result = verify_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=THREAD_CONFIG,
    )
    print(f"Q: {question}\nA: {extract_final_text(result['messages'][-1].content)}\n{'-' * 60}")

print(
    "Row 10 (cross-thread durable budget memory) cannot be tested locally — "
    "run mda dev . and re-ask the budget question in a NEW Studio thread instead."
)
